In [ ]:
! pip install -U accelerate bitsandbytes

In [ ]:
from transformers import PaliGemmaForConditionalGeneration , BitsAndBytesConfig , PaliGemmaProcessor
from PIL import Image
import torch
import requests
import PIL
import re
from torchvision.ops import box_convert
from torchvision.transforms.functional import to_pil_image , pil_to_tensor
from torchvision.utils import draw_bounding_boxes

In [ ]:
DEVICE = "cuda:0"
MODEL_ID = "google/paligemma-3b-mix-224"
skip_special_tokens = True
max_new_tokens = 500
do_sample = False

In [ ]:
def quantize_model():
  """model quantization to 4 bit """
  return BitsAndBytesConfig(load_4_bit = True)

In [ ]:
def load_model_processor(token,dtype,quantization_config_model,model_id:str = "google/paligemma-3b-mix-224"):
  """loading model and processor with hf_token for authorization"""
  model = PaliGemmaForConditionalGeneration.from_pretrained(
      model_id,
      quantization_config = quantization_config_model
  )
  processor = PaliGemmaProcessor.from_pretrained(model_id)
  return model , processor

In [ ]:
def load_image_from_url(url:str) -> Image:
  """ loading image from url and converting to PIL.Image object for futher processing"""
  image = Image.open(requests.get(url,stream=True).raw)
  return image

In [ ]:
def load_image(image_path:str) -> PIL.Image:
  image = Image.open(image_path)
  return image

In [ ]:
def inference_model(model,processor,image:Image.Image,prompt:str,do_sample:bool,skip_special_tokens:bool,max_new_tokens:int):
  """taking prompt and image and input with model , processor and other essential parameters to make inference"""
  model_inputs = processor(images=image,text=prompt,do_sample=do_sample,return_tensors="pt").to(model.device)
  model_ids = model_inputs["input_ids"].shape[-1]
  model = model.to(DEVICE).eval() # makeing model in evaluation mode
  with torch.inference_mode():
    generation = model.generate(**model_inputs,max_new_tokens=max_new_tokens)
    generation = generation[0][model_ids:]
    decoded_output = processor.decode(generation,skip_special_tokens=skip_special_tokens)
    print("[INFO] Model inference done")
  return decoded_output # containes the text in string format

In [ ]:
from typing import List,Tuple
def scaled_arranged_boxes(H:int,W:int,bbox:List) -> List:
  """ this function take list and input which contain bbox in [y_min,x_min,y_max,x_max] format and arranged and re notmalize it correctly"""
  scaled_bbox = [] # it is only working on on scale
  for i,val in enumerate(bbox):
    if i % 2 == 0: # y_axis
      scaled_bbox.append((val/1024) * H) # 1024 => 224 pixel size model in 14 patches
    else:
      scaled_bbox.append((val/1024)*W) # x axis

  return scaled_bbox


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
### ai generated >> not loving this part of matplotlib
def get_color_from_palette(number, min_val, max_val, cmap_name='viridis'):
    # 1. Normalize your number to a range between 0.0 and 1.0
    norm_number = (number - min_val) / (max_val - min_val)
    norm_number = max(0.0, min(1.0, norm_number)) # Clamp value

    # 2. Get the colormap
    cmap = plt.get_cmap(cmap_name)

    # 3. Get the color as RGBA (floats from 0-1)
    rgba_color = cmap(norm_number)

    # 4. Convert to 8-bit (0-255) RGB tuple
    rgb_color = tuple(int(x * 255) for x in rgba_color[:3])

    return rgb_color

# Example: Get color for number 75 out of a scale from 0 to 100
print(get_color_from_palette(75, min_val=0, max_val=100, cmap_name='plasma'))
# Output: (240, 78, 35)

In [ ]:
def remove_bbox_and_class(output,H:int,W:int):
  """ it takes the output from model either single class or multiclass and get the tensor of bounding box and class name out"""
  class_collection = output.split(";") # seperate class
  # example = [loc<xx>..... class_1 , loc<xxx>.....class_2]
  bboxes = []
  classes = []
  for val in class_collection:
    response = re.findall(r"\d+\.?\d*",val) # all values out
    bbox = [int(x) for x in response]
    bbox = scaled_arranged_boxes(H,W,bbox)
    bbox = torch.tensor(bbox)
    # print(bbox) >> debug
    classes.append(re.findall(r"\s\w*",val)[0].strip())
    bbox = bbox[...,[1,0,3,2]] ## tp convert to xyxy format
    bboxes.append(bbox)
  bboxes = torch.stack(bboxes)
  # print(bboxes)
  # print(classes) >> debug
  return bboxes,classes

In [ ]:
def return_color_pal(classes:List) -> List:
  """ taking list or classes as input and in basis getting color palette"""
  color_pal = []
  for i in range(len(classes)):
    color_pal.append(get_color_from_palette(i,min_val=0,max_val=100,cmap_name="plasma"))
  return color_pal

In [ ]:
string = "<loc1123><loc1234><loc1234><loc9876> cat;<loc1234>loc<0023><loc9876><loc8755> bat;<loc1234><loc2394><loc6871><loc6716> ball"
bboxes,classes = remove_bbox_and_class(string,640,480)
bboxes

In [ ]:
return_color_pal(classes)

In [ ]:
from IPython.display import display
def build_plot(image:Image.Image,bboxes:torch.Tensor,colors:List[Tuple],classes:List):
  """ take bboxes , colors , image , classes and build and plot"""
  if not isinstance(image,PIL.Image.Image):
    image = Image.open(image) # JUST some validation
  bounding_boxes = draw_bounding_boxes(
      image  = pil_to_tensor(image),
      labels=classes,
      boxes = bboxes,
      colors = colors,
      label_colors = colors,
      width = 4
  )
  display(to_pil_image(
      pic = bounding_boxes
  ))

In [ ]:
## inference pipeline
dtype = torch.bfloat16
url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/transformers/tasks/car.jpg?download=true"
quantization_config = quantize_model()
model,processor = load_model_processor(hf_token,dtype,quantization_config,MODEL_ID)
print("[INFO] model loading completed !!")
image = load_image_from_url(url)
print("[INFO] Image loaded !!!")
input_prompt = input("prompt: ")
outputs = inference_model(model,processor,image,input_prompt,do_sample,skip_special_tokens,max_new_tokens)
print("[INFO] Infernece completed !!")
if not isinstance(image,PIL.Image.Image):
  image = PIL.Image.open(image)
W,H = image.size
bboxes,classes = remove_bbox_and_class(outputs,H,W)
print("[INFO] Post processing completed")
colors = return_color_pal(classes)
print("[INFO] Color paletted loaded !!")
build_plot(image,bboxes,colors,classes)
print("[INFO] Plotting completed")

In [ ]:
image = load_image_from_url(url)
print("[INFO] Image loaded !!!")
input_prompt = input("prompt: ")
outputs = inference_model(model,processor,image,input_prompt,do_sample,skip_special_tokens,max_new_tokens)
print("[INFO] Infernece completed !!")
if not isinstance(image,PIL.Image.Image):
  image = PIL.Image.open(image)
W,H = image.size
bboxes,classes = remove_bbox_and_class(outputs,H,W)
print("[INFO] Post processing completed")
colors = return_color_pal(classes)
print("[INFO] Color paletted loaded !!")
build_plot(image,bboxes,colors,classes)
print("[INFO] Plotting completed")

In [ ]:
bboxes